# CNN rollout demo (step by step)

Watches the autoregressive **decode** procedure (`models.cnn.rollout`) one
placement at a time: encode the target sequence once, then drive the
`Stepper` cell by cell -- at each new cell the model predicts an op, we
sample one, place it, and the interpreter steps the IP -- building a program
in befunge execution order.

The model here is **untrained**, so the ops are essentially random and the
program won't reproduce the target. The point is to watch the rollout
mechanics proceed step by step; real programs come after training.

In [ ]:
import torch

from models.cnn.model import CNN
from models.cnn.rollout import rollout

torch.manual_seed(0)        # reproducible (untrained) model init
model = CNN().eval()

target = [3, 5, 7, 9]       # the sequence we want a program to print

`rollout` returns the final stepper, a stop status, and a `trace` of every
placement (cell, heading, op, and the program so far). Its `seed` controls
the op sampling (and the `?` op), so re-running with a different seed gives a
different rollout.

In [ ]:
s, status, trace = rollout(model, target, seed=45, max_places=120)

print(f'target: {target}\n')
for i, st in enumerate(trace):
    print(f"--- step {i}: IP ({st['x']},{st['y']}) heading {st['heading']} "
          f"-> place {st['op']!r} ---")
    print(st['program'])
    print('-'*47)
    print('\n\n\n')
print(f"\nstopped: {status} after {len(trace)} placements; "
      f"output {s.output!r}")

Each block shows the IP cell, the op sampled there, and the program so far --
the rollout building it one cell at a time, in the order the IP visits them.
Untrained, the ops are random and the output is junk; once `train.py` exists
the same loop should produce programs whose output matches the target's
leading terms. Change `seed` to watch a different rollout.